# Visualizing Snowflake Tables as a Graph

This is a brief but complete example of how to visualize graphs represented by tables in Snowflake, using the Graph Visualization for Python library for Neo4j.
The API for this is based on how one defines graph projections for the [Neo4j Graph Analytics for Snowflake application](https://neo4j.com/docs/snowflake-graph-analytics/current/).

## Setup

We will start by installing the necessary Python library requirements.

In [ ]:
%pip install neo4j-viz[snowflake]

We can now proceed to set up our connection to Snowflake by initializing a new session.
Please note that you may need more or fewer connection parameters depending on your Snowflake configuration.

In [ ]:
import os

from snowflake.snowpark import Session

# Configure according to your own setup
connection_parameters = {
    "account": os.environ.get("SNOWFLAKE_ACCOUNT"),
    "user": os.environ.get("SNOWFLAKE_USER"),
    "password": os.environ.get("SNOWFLAKE_PASSWORD"),
    "role": os.environ.get("SNOWFLAKE_ROLE"),
    "warehouse": os.environ.get("SNOWFLAKE_WAREHOUSE"),
}

session = Session.builder.configs(connection_parameters).create()

## Creating tables

In order to have something to visualize, we will now proceed to create a small example graph, represented by tables in Snowflake.
The first table we create will represent person nodes.

In [ ]:
session.sql(
    "CREATE OR REPLACE TABLE EXAMPLE_DB.DATA_SCHEMA.PERSONS (NODEID VARCHAR);"
).collect()

session.sql("""
INSERT INTO EXAMPLE_DB.DATA_SCHEMA.PERSONS VALUES
  ('Alice'),
  ('Bob'),
  ('Carol'),
  ('Dave'),
  ('Eve');
  """).collect()

The second table we create will also be one of nodes, but this time representing musical instruments.

In [ ]:
session.sql(
    "CREATE OR REPLACE TABLE EXAMPLE_DB.DATA_SCHEMA.INSTRUMENTS (NODEID VARCHAR);"
).collect()

session.sql("""
  INSERT INTO EXAMPLE_DB.DATA_SCHEMA.INSTRUMENTS VALUES
  ('Guitar'),
  ('Synthesizer'),
  ('Bongos'),
  ('Trumpet');
  """).collect()

In order to make a graph out of this, we should have some relations connecting nodes together.
The following table contains relationships from person to instrument nodes, representing that persons liking instruments.

In [ ]:
session.sql(
    "CREATE OR REPLACE TABLE EXAMPLE_DB.DATA_SCHEMA.LIKES (SOURCENODEID VARCHAR, TARGETNODEID VARCHAR);"
).collect()

session.sql("""
INSERT INTO EXAMPLE_DB.DATA_SCHEMA.LIKES VALUES
  ('Alice', 'Guitar'),
  ('Alice', 'Synthesizer'),
  ('Alice', 'Bongos'),
  ('Bob',   'Guitar'),
  ('Bob',   'Synthesizer'),
  ('Carol', 'Bongos'),
  ('Dave',  'Guitar'),
  ('Dave',  'Trumpet'),
  ('Dave',  'Bongos');
  """).collect()

## Creating the Visualization Graph

Now that we have our data set, we are ready to generate a `VisualizationGraph` that we can subsequently render.
We do so with a call go the `from_snowflake` convenience constructor of the `neo4j-viz` library.
Along with our Snowflake session, the input is a [project configuration](https://neo4j.com/docs/snowflake-graph-analytics/current/jobs/#jobs-project) that defines how we want the tables to be represented as a graph.
The project configuration syntax is the same as in the Neo4j Graph Analytics for Snowflake application.

In [ ]:
from neo4j_viz.snowflake import from_snowflake

VG = from_snowflake(
    session,
    {
        "nodeTables": [
            "EXAMPLE_DB.DATA_SCHEMA.PERSONS",
            "EXAMPLE_DB.DATA_SCHEMA.INSTRUMENTS",
        ],
        "relationshipTables": {
            "EXAMPLE_DB.DATA_SCHEMA.LIKES": {
                "sourceTable": "EXAMPLE_DB.DATA_SCHEMA.PERSONS",
                "targetTable": "EXAMPLE_DB.DATA_SCHEMA.INSTRUMENTS",
            }
        },
    },
)

To sanity check our visualization graph, let us count that the number of relationships is indeed nine.

In [ ]:
len(VG.relationships)

## Rendering the visualization

Let us now render our visualization graph as an interactive widget with `render_widget`, using only default render options.
The widget works in Jupyter environments such as JupyterLab, Notebook 7, VS Code and Colab, and provides two-way sync between Python and JavaScript.

In [ ]:
widget = VG.render_widget()
widget

The graph renders nicely, and we see that our two node types, "PERSONS" and "INSTRUMENTS", are colored and captioned differently.
By default, table names will determine both node and relationship captions, as well as the node coloring.
We can also see this in that relationships are rendered as arrows with the "LIKES" caption.

We can zoom in and out (mouse scroll-wheel), pan around, move nodes, and hover over nodes and relationships to see their properties.
The buttons on the top right also allow us to zoom, in addition to taking PNG snapshots of the graph.

## Customizing the visualization

If we are not completely satisfied with the graph is rendered, there are ways to customize it.
For example, we could change it so that every nodes gets its own color, by using the `color_nodes` method.
By passing it the `property` "SNOWFLAKEID", which will be unique for all nodes, it will give each node a new color.
We also make sure to set `override` to override the default coloring.

Since we are working with a widget, we can call `color_nodes` directly on it, and the change is synced to the visualization rendered above without having to render the graph again.

In [ ]:
widget.color_nodes(property="SNOWFLAKEID", override=True)

## Cleanup

Lastly, we lets clean up the tables we created and close our Snowflake session.

In [ ]:
session.sql("DROP TABLE IF EXISTS EXAMPLE_DB.DATA_SCHEMA.PERSONS").collect()
session.sql("DROP TABLE IF EXISTS EXAMPLE_DB.DATA_SCHEMA.INSTRUMENTS").collect()
session.sql("DROP TABLE IF EXISTS EXAMPLE_DB.DATA_SCHEMA.LIKES").collect()

session.close()